<div style="background-color: white; padding: 10px;">
<center>
    <img style="padding-left:15px"  height='50px' src="https://www.norkart.no/hubfs/norkart-logo-default.svg">
    <img style="padding-left:15px"  height='50px' src="https://www.kartverket.no/public/images/logo/kartverket-logo-large2.svg">
    <br />
    <img style="padding-right:15px" height='50px' src="https://kartai.no/wp-content/uploads/2025/03/cropped-KartAi-med-partnere-2048x1145.png">
    </center>
</div>

# ⛅ Cumulus Geographica - Lær hvordan bruke Cloud Native Geo (CNG)-teknologier? 🗺️

<a target="_blank" href="https://colab.research.google.com/github/kartAI/skygeo/blob/main/cumulus_geographica.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

------------
Hvordan unngår du hallusinering? Hvordan kan språkmodeller gjøre GIS-analyser?

Bli med på praktisk workshop der du lærer å kombinere kraften i moderne KI med geografiske data og analyser. I løpet av denne sesjonen vil du:

* Lære hvordan store språkmodeller (LLMs) kan transformere og effektivisere geografiske analyser
* Få hands-on erfaring med å koble ChatGPT-lignende modeller til PostGIS-databaser
* Utforske hvordan du kan stille komplekse geografiske spørsmål på naturlig språk
* Bygge interaktive kart og visualiseringer styrt av AI

Workshopen er designet for både nybegynnere og erfarne geomatikere som ønsker å utforske fremtidens analyseverktøy. Ta med laptop og bli med på å utforske der kunstig intelligens møter geografisk intelligens!

Ingen tidligere KI-erfaring nødvendig – bare ta med din geomatikkunnskap, laptop og god porsjon nysgjerrighet!
-------------




# Cumulus Geographica

* setup - duckdb, s3, httpfs, spatial
    * Forklaring på hva duckdb er
    * Linker til docs for duckdb

* Lakehouse-introduksjon - markdown
    * Overture Maps som eksempel
    * Andre åpne kilder? source.coop? Canada?

**Kick the tires**
* Overture maps - kick the tires. Places. 
    * Finn alle cafeer i en bbox og vis et kart med lonboard
    * Finn alle bygninger over 500 kvm
    * Kombiner tema-filtrering og fritekst-filtrering med bbox
**Mer avansert og kombinasjoner**
* Last inn kommunepolygon til en bestemt kommune fra en parquet-fil
* Last inn flomsoner innenfor bbox-en til kommunen over fra en parquet-fil
* Kirkebygg forenklet fra parquet - finne kirkebygg i kommunen og 10 km “buffer”. Hvilke kirker kan jeg bruke i beredskapssammenheng? “Hole kommune”. Hva er nærmeste vei til en kirke?

**Refleksjonsspørsmål**

**Tips og tricks!**
* Hvordan lage duckdb-sql med agenter og KI?
    * instructions-eksempel



#### ⚙️ Konfigurasjon og oppsett
Kjør cellene under.

In [19]:
%%capture
# load imports
%pip install dotenv geopandas folium matplotlib mapclassify duckdb
import os
import geopandas as gpd
import duckdb
from shapely import wkb
from pathlib import Path
from lonboard import Map, PathLayer, viz




#### ⛅ Hva er Cloud Native Geo-teknologi?

* GeoParquet
* Hive-partitioning `*` er magien!
* OvertureMaps som eksempel
* Linker til mer ressurser helt

In [ ]:
# setup duckdb connection
# Initialize DuckDB connection
conn = duckdb.connect(':memory:')

# Load spatial extension
conn.execute("LOAD spatial;")

# Set S3 region for Overture Maps
conn.execute("SET s3_region='us-west-2';")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

#### Kick the tires! 

Nå skal du prøve ut datauthenting fra OvertureMaps

1. Kjør cellen under
1. Se på dataschemaet ... http://....
1. Kan du hente ut kun ....


In [37]:
# Query pizza restaurants in NYC from Overture Maps
query = """
SELECT
    id,
    names.primary as name,
    confidence,
    CAST(socials AS JSON) as socials,
    geometry
FROM
    read_parquet('s3://overturemaps-us-west-2/release/2026-02-18.0/theme=places/type=place/*', 
                 filename=true, hive_partitioning=1)
WHERE
    categories.primary = 'pizza_restaurant'
    AND bbox.xmin BETWEEN -75 AND -73
    AND bbox.ymin BETWEEN 40 AND 41
"""

# Execute query
result_df = conn.sql(query).df()

display(result_df.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,id,name,confidence,socials,geometry
0,353d0312-fed0-41dd-ad3c-b20eb60ce3ec,Cherry Grove Pizza,0.980859,"[""https://www.facebook.com/693482650853096""]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
1,a1ac0990-d245-4c64-9b04-2609b2bf5e30,Grove Pizza,0.323479,"[""https://www.facebook.com/134080926636616""]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
2,a462844d-882a-4282-8301-920129a7637d,Papa John's Pizza,0.993309,"[""https://www.facebook.com/146172205446967""]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
3,0d4298fc-03ca-4abc-8cd7-252ad90a7cae,Rocco's Pizza,0.993792,"[""https://www.facebook.com/147024548709942""]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
4,4d698210-e9e2-4133-8958-bdfd53a4ab4f,Nonna's Pizzeria,0.970932,"[""https://www.facebook.com/1425133851123393""]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."


In [ ]:
# Create a duckdb table from the query
conn.execute("""
CREATE OR REPLACE TABLE pizza_restaurants AS
    SELECT
        id,
        names.primary AS name,
        confidence,
        CAST(socials AS JSON) AS socials,
        geometry
    FROM read_parquet(
        's3://overturemaps-us-west-2/release/2026-02-18.0/theme=places/type=place/*',
        filename=true,
        hive_partitioning=1
    )
    WHERE
        categories.primary = 'pizza_restaurant'
        AND bbox.xmin BETWEEN -75 AND -73
        AND bbox.ymin BETWEEN 40 AND 41
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
# Verify the geometry column exists and its type
r = conn.sql("DESCRIBE pizza_restaurants")
r.df()
# Or query just the column info
#conn.sql("SELECT column_name, data_type FROM information_schema.columns WHERE table_name='pizza_restaurants'").show()

,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,name,VARCHAR,YES,None,None,None
2,confidence,DOUBLE,YES,None,None,None
3,socials,VARCHAR,YES,None,None,None
4,geometry,BLOB,YES,None,None,None


In [ ]:
from shapely import wkt

# visualize with lonboard
query = conn.sql("select * from pizza_restaurants")
display(query.df().head())

df = query.df()

gdf = gpd.GeoDataFrame(
    df,
    geometry='geometry',
    crs="EPSG:4326"
)

#viz(query)

,id,name,confidence,socials,geometry
0,353d0312-fed0-41dd-ad3c-b20eb60ce3ec,Cherry Grove Pizza,0.980859,"[""https://www.facebook.com/693482650853096""]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
1,a1ac0990-d245-4c64-9b04-2609b2bf5e30,Grove Pizza,0.323479,"[""https://www.facebook.com/134080926636616""]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
2,a462844d-882a-4282-8301-920129a7637d,Papa John's Pizza,0.993309,"[""https://www.facebook.com/146172205446967""]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
3,0d4298fc-03ca-4abc-8cd7-252ad90a7cae,Rocco's Pizza,0.993792,"[""https://www.facebook.com/147024548709942""]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
4,4d698210-e9e2-4133-8958-bdfd53a4ab4f,Nonna's Pizzeria,0.970932,"[""https://www.facebook.com/1425133851123393""]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."


TypeError: Input must be valid geometry objects: bytearray(b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\xfc\xff8a\xc2ER\xc0\xf41\xc4VuTD@')

#### Filtreringer og analyser

1. Last inn norske kartdata - kun det du trenger videre. 
2. Se på data og visualiser

Kjør cellene under

In [ ]:
# Hent inn kommunepolygon for 1 kommune eller flere kommuner

Bruk kommunepolygon som filtrering videre!

In [ ]:
# bbox-filter fra kommunepolygon - deretter spatial filter med st_intersects



In [ ]:
# Hent ut alle flomsoner i kommunen

In [ ]:
# Hent ut alle kirkebygg i kommunen med buffer 10 km

#buffer 10 km - plot. 
## bbox og spatial filter - execute

In [ ]:
# Kun kirkebygg utenfor flomsone og med kapasitet over 100 personer

In [ ]:
#Export csv, tabell og plotkart + html-kart til disk